# Comparing 2 sets of cloud-based Particle data (EPILo and OMNI)

### demo, May 2026

This code demonstrates being able to read cloud-stored CDF files for 2 missions (without copying over or downloading locally), extracting fields into Pandas Dataframes, aligning the data, then sending to an algorithm.  We compare two particle time series from non-aligned datasets. Data processing uses fast parallel reads from S3 cloud storage. We then apply 2 algorithms: we compute the Pearson correlation coefficient between the paired series (linear association) and corresponding p-value, then a similar check with a moving lag window via FFT.  Scientists can cheerfully change the instruments and variables extracted, and send them to more appropriate algorithms.

For our demo, we use PSP/EPILo and OMNI.  We take up to 4 years of data (for the time range 09/2018-11/2022)  We are correlating PSP (which is moving) with OMNI (which is not), so the science is dubious, but the methods as as demo scaffold are sound. From EPILo, we extract proton count rates (H_CountRate) and sum into a single scalar value per timestamp by averaging across those dimensions, producing a 1D intensity-like time series. From the OMNI dataset, we extracts a scalar solar wind parameter (e.g., proton density) at 1-minute cadence. The two time series are then temporally aligned by pairing each EPILO measurement with the nearest OMNI measurement within a specified tolerance window, forming matched samples.

It takes about 1 minute to run per year of data (or per 600 EPILo files); the bulk of that is reading the files. In comparison, a laptop run of the same code is 2-4x slower. To improve performance we could do more sensible data extraction, cacheing of the intermediate time series, or switching to the HAPI protocol as viable options.  For the purposes of this demo, however, this works.


In [ ]:
# latest version, multiple file reads
from pathlib import Path
import pandas as pd
import numpy as np
import cdflib
from scipy.stats import pearsonr
from scipy.signal import correlate, correlation_lags
import cloudcatalog
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, as_completed
import re
import time
import gc
import matplotlib.pyplot as plt

def load_one_cdf(cdf_path, meta):
    cdf = cdflib.CDF(str(cdf_path))

    time = cdflib.cdfepoch.to_datetime(cdf.varget(meta['time_var']))
    data = np.asarray(cdf.varget(meta['data_var']), dtype=float)

    if data.ndim > 1:
        data = np.nanmean(data, axis=tuple(range(1, data.ndim)))

    if meta['fill']:
        try:
            attrs = cdf.varattsget(data_var)
            fillval = attrs.get("FILLVAL")
        except:
            fillval = None
        if fillval is not None:
            data[data == fillval] = np.nan

    n = min(len(time), len(data))

    return pd.DataFrame({
        "time": pd.to_datetime(time[:n], utc=True),
        meta['label']: data[:n],
    })

def _clean_concat(frames):
    if not frames:
        return None

    return (
        pd.concat(frames, ignore_index=True)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .drop_duplicates(subset="time")
        .sort_values("time")
        .reset_index(drop=True)
    )

def load_many_threaded(
    files,
    meta,
    noisy=False,
    max_workers=4,
    max_in_flight=None,
    batch_size=25,
):
    """
    Memory-bounded threaded replacement for serial load_many().

    Calls:
        load_one_cdf(path, meta)

    Key controls:
        max_workers: number of active reader threads
        max_in_flight: max submitted-but-not-collected futures
        batch_size: concatenate periodically instead of keeping all frames
    """

    files = list(files)
    icount = len(files)

    if icount == 0:
        raise ValueError("No files provided.")

    if max_in_flight is None:
        max_in_flight = max_workers * 2

    interval = max(1, int(icount / 20))

    partials = []
    batch = []
    pending = set()
    file_iter = iter(files)

    completed = 0

    def submit_next(pool):
        try:
            f = next(file_iter)
        except StopIteration:
            return False

        fut = pool.submit(load_one_cdf, f, meta)
        fut._source_path = f
        pending.add(fut)
        return True

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        # Prime only a bounded number of tasks
        for _ in range(min(max_in_flight, icount)):
            submit_next(pool)

        while pending:
            done, pending = wait(pending, return_when=FIRST_COMPLETED)

            for fut in done:
                path = getattr(fut, "_source_path", None)

                try:
                    df = fut.result()
                    if df is not None and len(df) > 0:
                        batch.append(df)
                except Exception as e:
                    if noisy:
                        print(f"Skipping {path}: {e}")

                completed += 1

                if completed % interval == 0:
                    print(f"... loaded {completed}/{icount} files ...")

                # Submit one new job only after one completed
                submit_next(pool)

                # Periodically collapse many small frames into one
                if len(batch) >= batch_size:
                    partial = _clean_concat(batch)
                    batch.clear()

                    if partial is not None and len(partial) > 0:
                        partials.append(partial)

                    gc.collect()

    if batch:
        partial = _clean_concat(batch)
        batch.clear()

        if partial is not None and len(partial) > 0:
            partials.append(partial)

        gc.collect()

    if not partials:
        raise ValueError("No files loaded successfully.")

    result = _clean_concat(partials)

    del partials
    gc.collect()

    if result is None or len(result) == 0:
        raise ValueError("No files loaded successfully.")

    return result

def load_many(files, meta):
    frames = []
    iskip = 0
    for f in files:
        try:
            df = load_one_cdf(f, meta)
            frames.append(df)
        except Exception as e:
            iskip += 1
            #print(f"Skipping {f}: {e}")

    if not frames:
        raise ValueError("No files loaded successfully.")

    if iskip > 0: print(f"(skipped {iskip} files)")

    return (
        pd.concat(frames, ignore_index=True)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .drop_duplicates(subset="time")
        .sort_values("time")
        .reset_index(drop=True)
    )

def fft_lag_corr_window(x, y, downstream, max_lag=60 * 24 * 7, min_lag=0):    
    """ downstream=True  -> x is downstream of y
        downstream=False -> y is downstream of x
    """
    
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    x = (x - x.mean()) / x.std()
    y = (y - y.mean()) / y.std()

    corr = correlate(y, x, mode="full", method="fft") if downstream else correlate(x, y, mode="full", method="fft")
    corr = corr / len(x)
    lags = correlation_lags(len(x), len(y), mode="full")

    keep = (lags >= min_lag) & (lags <= max_lag)

    return pd.DataFrame({
        "lag_samples": lags[keep],
        "r_approx": corr[keep],
    })

def fetch_two_timeseries(
    dataA_files, dataA_meta,
    dataB_files, dataB_meta,
    tolerance="5min",
    max_workers=6,
    max_in_flight=12,
    batch_size=25
):
    now = time.time()
    dataA_df = load_many_threaded(dataA_files, dataA_meta, False, max_workers, max_in_flight, batch_size)
    print(f"... took {time.time()-now} seconds to load {dataA_meta['label']} files")

    now = time.time()

    dataB_df = load_many_threaded(dataB_files, dataB_meta, False, max_workers, max_in_flight, batch_size)
    print(f"... took {time.time()-now} seconds to load {dataB_meta['label']} files")

    now = time.time()
    aligned = pd.merge_asof(
        dataA_df,
        dataB_df,
        on="time",
        direction="nearest",
        tolerance=pd.Timedelta(tolerance),
    ).dropna()
    print(f"... took {time.time()-now} seconds to align data") 
    if len(aligned) < 3:
        raise ValueError(
            f"Only {len(aligned)} aligned samples found. "
            "Check date overlap or increase tolerance."
        )

    A = aligned[dataA_meta["label"]].to_numpy()
    B = aligned[dataB_meta["label"]].to_numpy()

    return {
        "n_"+dataA_meta["label"]: len(dataA_df),
        "n_"+dataB_meta["label"]: len(dataB_df),
        "n_aligned": len(aligned),
        "data_A": A,
        "data_B": B
    }

def do_calcs(A, B, downstream=True):
    # sample calc 1: Pearson coefficient
    now = time.time()
    r, p = pearsonr(A, B)
    print(f"... Pearson calc took {time.time()-now} seconds for computation")

    # sample calc 2: Pearson with lags
    now = time.time()
    lags = fft_lag_corr_window(A, B, downstream)

    print(f"... Lag calc took {time.time()-now} seconds for computation")
    
    return {
        "pearson_r": r,
        "p_value": p,
        "lags": lags
    }

def plot_fft_lag(df_fft, dt_seconds=60):
    # Convert lag to hours
    lag_hours = df_fft["lag_samples"] * dt_seconds / 3600.0
    # Mark strongest correlation
    idx = df_fft["r_approx"].abs().idxmax()
    best_lag_hr = lag_hours.iloc[idx]
    best_r = df_fft["r_approx"].iloc[idx]
    print(f"Best lag: {best_lag_hr:.2f} hours, r ≈ {best_r:.5f}")
    plt.figure()
    plt.plot(lag_hours, df_fft["r_approx"])
    plt.axhline(0)
    plt.axvline(0)
    plt.scatter([best_lag_hr], [best_r])
    plt.xlabel("Lag (hours)")
    plt.ylabel("Cross-correlation (approx)")
    plt.title("FFT Lagged Cross-Correlation")
    plt.grid()
    plt.show()
   
# main driver

# 1) CHOOSE TIME WINDOWS
start= "2018-09-29T00:00:00Z"
stops = ["2018-10-10T00:00:00Z","2019-09-29T00:00:00Z","2022-11-14T00:00:00Z"]
stop = stops[1] # 0 = 11 days, ~sec to read; 1 = 1 year, ~1 min to read; 2 = 4 years, ~5 min to read

# 2) FETCH DATA LISTINGS, THEN FETCH DATA
print("Fetching filelists...")
fr = cloudcatalog.CloudCatalog("s3://gov-nasa-hdrl-data1/")
dataidA, dataidB = 'PSP_ISOIS-EPILO_L2-IC', 'OMNI_HRO2_1MIN'
filekeys1 = fr.request_cloud_catalog(dataidA,start_date=start,stop_date=stop)
epilo_files = filekeys1['datakey'].to_list()
filekeys2 = fr.request_cloud_catalog(dataidB,start_date=start,stop_date=stop)
omni_files = filekeys2['datakey'].tolist()
print(f"Len of epilo is {len(epilo_files)}, omniweb is {len(omni_files)}")
now = time.time()
epilo_metadata = {"time_var": "Epoch_ChanR",
                  "data_var": "H_CountRate_ChanR",
                  "label" : "epilo",
                  "fill": False}
omni_metadata = {"time_var": "Epoch",
                 "data_var": "proton_density",
                 "label" : "omniweb",
                 "fill": True}
datasets = fetch_two_timeseries(
    epilo_files, epilo_metadata,
    omni_files, omni_metadata,
    tolerance="5min",
    max_workers=16,
    max_in_flight=32,
    batch_size=64
)

# 3) ALGORITHMS AND RESULTS

result = do_calcs(datasets["data_A"], datasets["data_B"], downstream=False)

print(f"\nElapsed time: {time.time()-now} seconds")
print("EPILO samples:", datasets["n_epilo"])
print("OMNIWeb samples:", datasets["n_omniweb"])
print("Aligned samples:", datasets["n_aligned"])
print("Pearson r:", result["pearson_r"])
print("p-value:", result["p_value"])
plot_fft_lag(result["lags"])
